# Genotype–Phenotype Heterogeneity Dataset Exploration with `mlcroissant`

This notebook provides a guided exploration of the FAIR^2 dataset: *Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency*, using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset is structured by a Croissant schema and available at:

```
https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json
```

The schema describes multiple record sets capturing clinical features, hormone levels, adrenal imaging, and genetic variants for three female patients with non-classical 21-hydroxylase deficiency.

In [ ]:
# Ensure mlcroissant library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json'

# Load metadata and records
dataset = mlc.Dataset(url)

# Access metadata as an object
metadata_obj = dataset.metadata
print(f"{metadata_obj.name}: {metadata_obj.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id` values.

This provides context for which tables and fields are available, each uniquely referenced by their `@id`.

In [ ]:
# List all record sets defined in the metadata, along with their fields
record_sets = list(dataset.record_sets())  # returns list of RecordSet objects

for rs in record_sets:
    print(f"Record Set: {rs.name}")
    print(f"  @id: {rs.id}")
    print(f"  Description: {rs.description}")
    print(f"  Fields:")
    for field in rs.fields:
        print(f"    - {field.name} (@id: {field.id}, dataType: {field.dataType})")
    print()
# Pick a record set for demonstration
if len(record_sets) > 0:
    primary_record_set = record_sets[0]  # Use the first record set for the following steps
    print(f"Example Record Set chosen: {primary_record_set.name} (@id: {primary_record_set.id})")

## 3. Data Extraction
Load data from each record set into pandas DataFrames for further analysis.

***All entities (record sets, fields, columns) are referenced by their `@id`.***

In [ ]:
# Extract all record sets by @id into DataFrames
dataframes = {}
record_set_ids = [rs.id for rs in record_sets]

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df

# Display the columns for the primary record set
primary_rs_id = primary_record_set.id
print(f"Columns for record set '{primary_record_set.name}' (@id: {primary_rs_id}):")
print(dataframes[primary_rs_id].columns.tolist())

# Show sample data
dataframes[primary_rs_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common processing steps: filter, normalize, and group data.

Below, we select a numeric field and demonstrate filtering, normalization, and grouping. 

All fields are referenced by their `@id`.

In [ ]:
# Choose a numeric field for EDA
# Find a numeric field in the primary record set
numeric_field_id = None
for field in primary_record_set.fields:
    if field.dataType in ['schema:Float', 'schema:Integer', 'schema:Number']:
        numeric_field_id = field.id
        print(f"Using numeric field: {field.name} (@id: {numeric_field_id}, dataType: {field.dataType})")
        break

if numeric_field_id and numeric_field_id in dataframes[primary_rs_id].columns:
    # Filter records
    threshold = 10
    filtered_df = dataframes[primary_rs_id][dataframes[primary_rs_id][numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize the numeric field
    normalized_col = f"{numeric_field_id}_normalized"
    filtered_df[normalized_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, normalized_col]].head())

    # Choose a categorical/grouping field
    group_field_id = None
    for field in primary_record_set.fields:
        if field.dataType in ['schema:Text', 'schema:Boolean'] and field.id != numeric_field_id:
            group_field_id = field.id
            print(f"Grouping by: {field.name} (@id: {group_field_id}, dataType: {field.dataType})")
            break

    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped summary (mean {numeric_field_id}) by {group_field_id}:")
        print(grouped_df.head())
else:
    print("No numeric field found in the selected record set for demonstration.")

## 5. Visualization
Visualize distributions or relationships between numeric and categorical fields.

In [ ]:
# Visualization: numeric distribution and grouped bar chart
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id and numeric_field_id in dataframes[primary_rs_id].columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(dataframes[primary_rs_id][numeric_field_id].dropna(), bins=10, kde=True)
    plt.title(f"Distribution of {numeric_field_id} in {primary_record_set.name}")
    plt.xlabel(numeric_field_id)
    plt.show()

    if group_field_id and group_field_id in dataframes[primary_rs_id].columns:
        plt.figure(figsize=(8, 4))
        sns.barplot(
            x=group_field_id, y=numeric_field_id, data=dataframes[primary_rs_id]
        )
        plt.title(f"Mean {numeric_field_id} by {group_field_id} in {primary_record_set.name}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()

## 6. Conclusion

- **Data Structure:** The FAIR^2 dataset is described by a Croissant schema and includes multiple record sets covering clinical, laboratory, imaging, and genetic data for three patients.
- **Exploration:** Using `mlcroissant`, data was loaded and explored record set by record set, referencing all entities by their `@id`.
- **EDA:** Demonstrated filtering, normalization, and grouping on numeric fields.
- **Visualization:** Showed distributions and relationships among clinical features.

### Limitations
- Extremely small cohort (N=3) limits generalizability.
- Dataset primarily supports clinical illustration and academic research; not suitable for AI training.

Explore further by selecting different record sets and fields by their `@id`!
